# UCS420: Cognitive Computing — Lab Assignment 9
## Part A: Cognitive Assistant for Reading Medicine-Bottle Labels (vision-based)

**Name:** Divyanshi Dhawan*Roll No:**1024160057

### 1. Problem and design
Many people (older adults, people with low vision or memory difficulties) struggle to read small print on medicine bottles. This assistant **perceives** a photo of a label, **understands** it (OCR + information extraction), **reasons** about it (is it expired? which medicine is this?) and **interacts** by reading a plain-language summary aloud.

```
 Image ──► Pre-processing ──► OCR (EasyOCR) ──► Field extraction ──► Reasoning ──► Spoken / text summary
 (photo)   grayscale, CLAHE,   text + confidence  regex + fuzzy match   expiry check,    (gTTS audio)
           denoise, threshold  per text line       to drug database      drug info
```

| Cognitive stage | Component in this notebook |
|---|---|
| Perception | Image enhancement + EasyOCR |
| Understanding | Regex field extraction (strength, batch, expiry, directions) + fuzzy drug-name matching |
| Reasoning | Expiry check, brand → generic lookup, confidence flags |
| Interaction | Text summary + text-to-speech |

> **Safety note:** this is a prototype for education. It must never replace a pharmacist or doctor; the assistant always asks the user to verify.

## 2. Setup

In [ ]:
!pip -q install easyocr rapidfuzz gTTS

In [ ]:
import re, os, glob, random, calendar, datetime as dt
import numpy as np, cv2
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw, ImageFont, ImageFilter
from rapidfuzz import process, fuzz
import easyocr, torch

USE_GPU = torch.cuda.is_available()
print("GPU available:", USE_GPU)
reader = easyocr.Reader(['en'], gpu=USE_GPU)      # downloads the model on first run

## 3. Dataset
Pick **one** option (Option C always works and is used to measure accuracy because it has ground truth):

* **Option A – your own photos:** upload photos of medicine bottles / strips to `/content/labels/`.
* **Option B – dataset from the internet:** search Kaggle / Roboflow Universe / Google Dataset Search for *"medicine label OCR"* or *"pill bottle label"* datasets, download the zip, and extract into `/content/labels/`. For Kaggle set your dataset slug below (needs `kaggle.json`).
* **Option C – synthetic labels (generated below):** printed-style labels with known text, so extraction accuracy can be measured.

In [ ]:
os.makedirs('/content/labels', exist_ok=True)

# ---- Option A: upload from your computer (uncomment) ----
# from google.colab import files
# for name, data in files.upload().items():
#     open(f'/content/labels/{name}', 'wb').write(data)

# ---- Option B: Kaggle dataset (uncomment, fill in the slug of the dataset you chose) ----
# from google.colab import files; files.upload()                     # upload kaggle.json
# !mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
# DATASET_SLUG = "owner/dataset-name"
# !kaggle datasets download -d $DATASET_SLUG -p /content/labels --unzip

user_images = sorted(sum([glob.glob(f'/content/labels/**/*.{e}', recursive=True) for e in ('jpg', 'jpeg', 'png')], []))
print(len(user_images), "user/downloaded images found")

### Option C – synthetic label generator (with ground truth)

In [ ]:
def get_font(size, bold=False):
    names = ["DejaVuSans-Bold.ttf" if bold else "DejaVuSans.ttf",
             "/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf" if bold else "/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf"]
    for n in names:
        try: return ImageFont.truetype(n, size)
        except Exception: pass
    return ImageFont.load_default()

SYNTH_TRUTH = [
    dict(name="Paracetamol", strength="500 mg", batch="PX2417", mfg="01/2025", exp="12/2027",
         directions="Take 1 tablet every 6 hours after food", maker="MediCure Pharma Ltd."),
    dict(name="Amoxicillin", strength="250 mg", batch="AMX9086", mfg="03/2024", exp="02/2026",
         directions="Take 1 capsule three times daily", maker="HealWell Labs"),
    dict(name="Metformin", strength="850 mg", batch="MF5521B", mfg="06/2025", exp="05/2028",
         directions="Take 1 tablet twice daily with meals", maker="Nova Biotech"),
    dict(name="Cetirizine", strength="10 mg", batch="CT7730", mfg="09/2025", exp="08/2027",
         directions="Take 1 tablet once daily at night", maker="Sunrise Remedies"),
]

def make_label(t, noisy=False, seed=0):
    rnd = random.Random(seed)
    W, H = 900, 520
    img = Image.new("RGB", (W, H), (250, 248, 240))
    d = ImageDraw.Draw(img)
    d.rectangle([10, 10, W-10, H-10], outline=(30, 60, 140), width=5)
    d.text((40, 30), t["name"].upper(), font=get_font(64, True), fill=(20, 40, 120))
    d.text((40, 120), f"Tablets IP  {t['strength']}", font=get_font(40), fill=(0, 0, 0))
    d.text((40, 200), t["directions"], font=get_font(28), fill=(0, 0, 0))
    d.text((40, 270), f"Batch No: {t['batch']}", font=get_font(32), fill=(0, 0, 0))
    d.text((40, 330), f"Mfg: {t['mfg']}", font=get_font(32), fill=(0, 0, 0))
    d.text((40, 390), f"EXP: {t['exp']}", font=get_font(36, True), fill=(160, 0, 0))
    d.text((40, 450), f"Manufactured by {t['maker']}", font=get_font(24), fill=(60, 60, 60))
    arr = np.array(img)
    if noisy:
        arr = cv2.GaussianBlur(arr, (3, 3), 0)
        arr = np.clip(arr + np.random.RandomState(seed).normal(0, 14, arr.shape), 0, 255).astype(np.uint8)
        M = cv2.getRotationMatrix2D((W/2, H/2), rnd.uniform(-4, 4), 1.0)
        arr = cv2.warpAffine(arr, M, (W, H), borderValue=(250, 248, 240))
    return arr[:, :, ::-1].copy()          # RGB -> BGR (OpenCV style)

synthetic = [make_label(t, noisy=(i % 2 == 1), seed=i) for i, t in enumerate(SYNTH_TRUTH)]
fig, ax = plt.subplots(1, 4, figsize=(18, 4))
for a, im, t in zip(ax, synthetic, SYNTH_TRUTH):
    a.imshow(im[:, :, ::-1]); a.set_title(t["name"]); a.axis('off')
plt.show()

## 4. Perception — image pre-processing

In [ ]:
def preprocess(img_bgr, scale=2.0):
    """Enhance a label photo for OCR: upscale -> grayscale -> denoise -> CLAHE contrast -> adaptive threshold."""
    img = cv2.resize(img_bgr, None, fx=scale, fy=scale, interpolation=cv2.INTER_CUBIC)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    gray = cv2.fastNlMeansDenoising(gray, None, h=10)
    gray = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8)).apply(gray)
    return cv2.adaptiveThreshold(gray, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY, 31, 15)

demo = synthetic[1]
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].imshow(demo[:, :, ::-1]); ax[0].set_title("Original (noisy)"); ax[0].axis('off')
ax[1].imshow(preprocess(demo), cmap='gray'); ax[1].set_title("Pre-processed"); ax[1].axis('off')
plt.show()

## 5. Perception — OCR

In [ ]:
def run_ocr(img):
    """Return a list of detected lines: {text, conf, h} sorted top-to-bottom."""
    out = []
    for box, text, conf in reader.readtext(img):
        ys = [p[1] for p in box]
        out.append({"text": text.strip(), "conf": float(conf), "h": max(ys) - min(ys), "y": min(ys)})
    return sorted(out, key=lambda r: r["y"])

lines = run_ocr(synthetic[0])
for l in lines: print(f"{l['conf']:.2f}  {l['text']}")

## 6. Understanding — field extraction

In [ ]:
# Small knowledge base: generic name -> (purpose).  Brand names map to a generic.
DRUG_INFO = {
    "paracetamol": "pain reliever and fever reducer",
    "ibuprofen": "anti-inflammatory pain reliever (NSAID)",
    "amoxicillin": "antibiotic for bacterial infections",
    "azithromycin": "antibiotic for bacterial infections",
    "metformin": "lowers blood sugar in type 2 diabetes",
    "cetirizine": "antihistamine for allergy symptoms",
    "atorvastatin": "lowers cholesterol",
    "omeprazole": "reduces stomach acid",
    "pantoprazole": "reduces stomach acid",
    "diclofenac": "anti-inflammatory pain reliever (NSAID)",
    "amlodipine": "lowers blood pressure",
    "losartan": "lowers blood pressure",
}
BRANDS = {"crocin": "paracetamol", "dolo": "paracetamol", "calpol": "paracetamol", "panadol": "paracetamol",
          "brufen": "ibuprofen", "augmentin": "amoxicillin", "azithral": "azithromycin",
          "glycomet": "metformin", "pan": "pantoprazole", "voveran": "diclofenac"}
NAMES = list(DRUG_INFO) + list(BRANDS)

RE_STRENGTH = re.compile(r"(\d+(?:\.\d+)?)\s*(mg|mcg|µg|ml|iu|g)\b", re.I)
RE_EXPIRY   = re.compile(r"(?:EXP(?:IRY|IRES)?|USE\s*BEFORE)\.?\s*(?:DATE)?\s*[:.\-]?\s*"
                         r"(\d{1,2}\s*[/\-.]\s*\d{2,4}(?:\s*[/\-.]\s*\d{2,4})?|[A-Za-z]{3,9}\.?\s*\d{2,4})", re.I)
RE_MFG      = re.compile(r"(?:MFG|MANUFACTURED?|MFD)\.?\s*(?:DATE)?\s*[:.\-]?\s*(\d{1,2}\s*[/\-.]\s*\d{2,4}|[A-Za-z]{3,9}\.?\s*\d{2,4})", re.I)
RE_BATCH    = re.compile(r"(?:BATCH\s*(?:NO\.?|NUMBER)?|B\.?\s?NO\.?|LOT)\s*[:.#\-]?\s*([A-Z0-9][A-Z0-9\-]{2,})", re.I)
RE_DIRECT   = re.compile(r"\b(take|tablet|capsule|daily|twice|once|every|hours?|before|after|meals?|apply|dose)\b", re.I)

def parse_date(s):
    """'12/2027', '12/27', 'Dec 2027', '15/12/2027' -> datetime.date (end of month when day is not given)."""
    s = re.sub(r"\s+", "", s.strip()).replace('.', '/').replace('-', '/')
    m = re.fullmatch(r"(\d{1,2})/(\d{2}|\d{4})", s)
    if m:
        mo, yr = int(m[1]), int(m[2]); yr += 2000 if yr < 100 else 0
        if 1 <= mo <= 12: return dt.date(yr, mo, calendar.monthrange(yr, mo)[1])
    m = re.fullmatch(r"(\d{1,2})/(\d{1,2})/(\d{2}|\d{4})", s)
    if m:
        d_, mo, yr = int(m[1]), int(m[2]), int(m[3]); yr += 2000 if yr < 100 else 0
        try: return dt.date(yr, mo, d_)
        except ValueError: return None
    m = re.fullmatch(r"([A-Za-z]{3})[A-Za-z]*/?(\d{2}|\d{4})", s)
    if m:
        mons = {n.lower(): i for i, n in enumerate(calendar.month_abbr) if n}
        mo = mons.get(m[1].lower()); yr = int(m[2]); yr += 2000 if yr < 100 else 0
        if mo: return dt.date(yr, mo, calendar.monthrange(yr, mo)[1])
    return None

OCR_FIX = str.maketrans({"0": "o", "1": "l", "5": "s", "4": "a", "3": "e", "8": "b", "6": "g"})   # common OCR digit/letter swaps

def match_drug(lines):
    """Fuzzy-match OCR tokens against the drug list (tolerating digit/letter confusions); fall back to the largest printed text."""
    best = (None, 0)
    for l in lines:
        for tok in re.findall(r"[A-Za-z0-9]{4,}", l["text"]):
            if sum(ch.isalpha() for ch in tok) < 3: continue
            for variant in {tok.lower(), tok.lower().translate(OCR_FIX), tok.lower().replace("1", "i").translate(OCR_FIX)}:
                r = process.extractOne(variant, NAMES, scorer=fuzz.ratio, score_cutoff=80)
                if r and r[1] > best[1]: best = (r[0], r[1])
    if best[0]:
        generic = BRANDS.get(best[0], best[0])
        return {"printed": best[0], "generic": generic, "match_score": best[1], "purpose": DRUG_INFO.get(generic)}
    big = max(lines, key=lambda l: l["h"]) if lines else None
    return {"printed": big["text"] if big else None, "generic": None, "match_score": 0, "purpose": None}

def extract_info(lines):
    text = "\n".join(l["text"] for l in lines)
    exp = RE_EXPIRY.search(text); mfg = RE_MFG.search(text); bat = RE_BATCH.search(text)
    return {
        "drug": match_drug(lines),
        "strengths": sorted({f"{n} {u.lower()}" for n, u in RE_STRENGTH.findall(text)}),
        "expiry_raw": exp.group(1).strip() if exp else None,
        "expiry": parse_date(exp.group(1)) if exp else None,
        "mfg_raw": mfg.group(1).strip() if mfg else None,
        "batch": bat.group(1).upper() if bat else None,
        "directions": [l["text"] for l in lines if RE_DIRECT.search(l["text"]) and not RE_BATCH.search(l["text"])],
        "mean_conf": float(np.mean([l["conf"] for l in lines])) if lines else 0.0,
    }

## 7. Reasoning and interaction

In [ ]:
def expiry_status(exp, today=None):
    today = today or dt.date.today()
    if exp is None: return "unknown", "I could not read the expiry date."
    days = (exp - today).days
    if days < 0:  return "expired", f"This medicine expired on {exp:%d %B %Y}. Please do NOT use it."
    if days <= 60: return "soon",   f"This medicine expires soon, on {exp:%d %B %Y}."
    return "ok", f"The expiry date is {exp:%d %B %Y}, so it is still in date."

def build_summary(info):
    d = info["drug"]; parts = []
    if d["generic"]:
        name = d["generic"].title() + (f" (sold as {d['printed'].title()})" if d["printed"] != d["generic"] else "")
        art = "an" if d["purpose"] and d["purpose"][0] in "aeiou" else "a"
        parts.append(f"This looks like {name}" + (f", {art} {d['purpose']}." if d["purpose"] else "."))
    elif d["printed"]:
        parts.append(f"The largest text reads '{d['printed']}', but I could not match it to a known medicine.")
    if info["strengths"]: parts.append("Strength: " + ", ".join(info["strengths"]) + ".")
    if info["directions"]: parts.append("Directions printed on the label: " + "; ".join(info["directions"]) + ".")
    if info["batch"]: parts.append(f"Batch number {info['batch']}.")
    status, msg = expiry_status(info["expiry"]); parts.append(msg)
    if info["mean_conf"] < 0.5: parts.append("The photo was hard to read, so please retake it in better light.")
    parts.append("Always confirm with your pharmacist or doctor before taking any medicine.")
    return " ".join(parts), status

def speak(text, fname="/content/summary.mp3"):
    try:
        from gtts import gTTS
        from IPython.display import Audio, display
        gTTS(text, lang='en').save(fname); display(Audio(fname))
    except Exception as e:
        print("(text-to-speech unavailable:", e, ")")

def analyze_label(img_bgr, ocr=None):
    """Full pipeline. Runs OCR on the raw and the pre-processed image and keeps the more confident result."""
    ocr = ocr or run_ocr
    best = None
    for tag, im in (("raw", img_bgr), ("preprocessed", preprocess(img_bgr))):
        lines = ocr(im)
        score = np.mean([l["conf"] for l in lines]) if lines else 0
        if best is None or score > best[0]: best = (score, tag, lines)
    _, tag, lines = best
    info = extract_info(lines); info["ocr_input"] = tag
    return info, lines

## 8. Run the assistant

In [ ]:
def show_result(img, info, summary, status):
    plt.figure(figsize=(7, 4)); plt.imshow(img[:, :, ::-1]); plt.axis('off')
    plt.title({"ok": "In date", "soon": "Expires soon", "expired": "EXPIRED", "unknown": "Expiry unknown"}[status]); plt.show()
    print("OCR input used:", info["ocr_input"], "| mean confidence:", round(info["mean_conf"], 2))
    print("Drug     :", info["drug"])
    print("Strength :", info["strengths"], "| Batch:", info["batch"], "| Expiry:", info["expiry_raw"], "->", info["expiry"])
    print("\nSPOKEN SUMMARY:\n" + summary)

for img in synthetic:
    info, lines = analyze_label(img)
    summary, status = build_summary(info)
    show_result(img, info, summary, status)
    speak(summary)
    print("=" * 90)

Run on your own / downloaded images (Option A/B):

In [ ]:
for path in user_images[:5]:
    img = cv2.imread(path)
    info, lines = analyze_label(img)
    summary, status = build_summary(info)
    print(path); show_result(img, info, summary, status); speak(summary); print("=" * 90)

## 9. Evaluation on the synthetic set (ground truth available)

In [ ]:
def evaluate(truth_list, images, ocr=None):
    rows = []
    for t, img in zip(truth_list, images):
        info, _ = analyze_label(img, ocr=ocr)
        rows.append({
            "label": t["name"],
            "name_ok":     (info["drug"]["generic"] or "").lower() == t["name"].lower(),
            "strength_ok": t["strength"].lower() in [s.lower() for s in info["strengths"]],
            "batch_ok":    (info["batch"] or "") == t["batch"],
            "expiry_ok":   info["expiry_raw"] is not None and parse_date(info["expiry_raw"]) == parse_date(t["exp"]),
            "mean_conf":   round(info["mean_conf"], 2),
        })
    return rows

import pandas as pd
res = pd.DataFrame(evaluate(SYNTH_TRUTH, synthetic))
display(res)
print("Field-level accuracy:"); print(res[["name_ok", "strength_ok", "batch_ok", "expiry_ok"]].mean().round(2))

## 10. Results, limitations and discussion (write your own observations)

* **What worked:** _e.g. printed labels with good contrast were read with high confidence; fuzzy matching recovered names with 1–2 wrong characters._
* **Where it failed:** _e.g. curved bottles, glare, small print and low light reduce OCR accuracy; batch numbers with O/0 and I/1 confusion._
* **Effect of pre-processing:** compare the `OCR input used` field (raw vs preprocessed) across images.
* **Safety / ethics:** the assistant is advisory only; it never recommends doses itself, only repeats what is printed, and always asks the user to verify with a professional. A wrong reading could be harmful, so low-confidence results trigger a "retake the photo" prompt.
* **Future work:** bottle-curvature unwarping, text-detection fine-tuning on a medicine-label dataset, a larger drug database (e.g. a national drug list), multilingual labels, and a phone-camera front end.